# NUS ST5221 — Stochastic Processes and Applications
## Real-world case study: Capital Bikeshare demand as a stochastic system

This notebook develops a **single coherent case study** around the UCI Bike Sharing dataset and uses it to study major ST5221 ideas:

- stochastic processes and sample paths;
- discrete-time Markov chains;
- communication, irreducibility, periodicity and stationary distributions;
- $n$-step transitions and mixing;
- first-passage / hitting-time calculations;
- continuous-time Markov chain (CTMC) generators;
- Kolmogorov evolution via $P(t)=e^{Qt}$;
- Poisson count models and why a homogeneous Poisson assumption can fail;
- renewal processes and renewal-reward reasoning;
- Brownian motion as a scaling limit of accumulated innovations;
- reflection-principle / first-passage simulation.

The notebook deliberately separates **what is directly observed** from **what is a modelling approximation**.

### Dataset

UCI Bike Sharing Dataset: Capital Bikeshare hourly rental counts for 2011–2012, with weather and calendar variables.

Official source:

https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

The hourly file has 17,379 rows.

---

## Why this dataset is useful for stochastic-process thinking

Let $X_t$ denote a stochastic description of bike demand at hour $t$.

We can ask several different questions.

1. **Regime dynamics:** if demand is currently Low / Medium / High, how likely is each regime one hour later?
2. **Long-run behaviour:** what fraction of time does the demand-regime chain spend in each regime?
3. **First passage:** how long does it typically take to hit a High-demand state?
4. **Continuous-time approximation:** what transition-rate generator $Q$ is implied by the observed regime changes?
5. **Count process:** can hourly rental counts be approximated as Poisson counts with a time-varying rate?
6. **Renewal:** how regularly do High-demand episodes begin?
7. **Diffusion limit:** after removing predictable structure, does accumulated demand noise behave qualitatively like a Brownian scaling limit?

These are different stochastic models applied to different aspects of the same physical system.

# 1. Architecture and design patterns

The code is intentionally modular.

### Strategy pattern

Two places use interchangeable strategies:

- `DatasetSource`: remote UCI ZIP versus an already-downloaded local CSV;
- `DemandRegimeStrategy`: the rule used to discretize demand into stochastic states.

This lets us replace the data source or state-definition algorithm without changing downstream analysis.

### Factory / Registry pattern

`AnalysisRegistry` maps short names to analyser classes. This is useful when the notebook grows to multiple stochastic models.

### Facade pattern

`PlotFactory` provides a small stable plotting API over Bokeh. Statistical code does not need to know Bokeh details.

### Immutable configuration

`NotebookConfig` stores experiment parameters in one place.

The aim is not to use patterns for their own sake. The architecture separates:

$$
\text{data acquisition}
\rightarrow
\text{state construction}
\rightarrow
\text{stochastic model}
\rightarrow
\text{diagnostics}
\rightarrow
\text{visualisation}.
$$

> **Patched edition.** This version includes compatibility and robustness fixes for modern pandas/Bokeh/statsmodels environments: stable categorical levels for formula prediction, warning-free Bokeh tables, resilient dataset acquisition with local caching/fallback, direct hourly-gap checks, defensive stochastic-model validation, and memory-efficient Brownian first-passage simulation.

In [1]:
# If your environment is missing packages, uncomment:
# %pip install pandas numpy scipy statsmodels scikit-learn bokeh networkx

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, Iterable, Optional, Sequence, Type

import io
import math
import urllib.request
import urllib.error
import zipfile

import numpy as np
import pandas as pd

from scipy import linalg, stats
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error

import statsmodels.api as sm
import statsmodels.formula.api as smf

import networkx as nx

from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import (
    ColumnDataSource,
    DataTable,
    Div,
    HoverTool,
    LinearColorMapper,
    ColorBar,
    TableColumn,
)
from bokeh.palettes import Viridis256
from bokeh.plotting import figure

output_notebook()

RNG = np.random.default_rng(42)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

Loading BokehJS ...

In [2]:
@dataclass(frozen=True)
class NotebookConfig:
    random_seed: int = 42
    cache_dir: str = "./data"
    dataset_url: str = (
        "https://archive.ics.uci.edu/static/public/275/"
        "bike%2Bsharing%2Bdataset.zip"
    )
    dataset_fallback_csv_url: str = (
        "https://raw.githubusercontent.com/udacity/deep-learning/"
        "master/first-neural-network/Bike-Sharing-Dataset/hour.csv"
    )
    download_timeout_seconds: int = 30
    regime_quantiles: tuple[float, float] = (0.33, 0.67)
    chronological_train_fraction: float = 0.80
    high_episode_quantile: float = 0.90
    brownian_mc_paths: int = 5_000
    brownian_mc_steps: int = 500

CFG = NotebookConfig()
CFG

NotebookConfig(random_seed=42, cache_dir='./data', dataset_url='https://archive.ics.uci.edu/static/public/275/bike%2Bsharing%2Bdataset.zip', dataset_fallback_csv_url='https://raw.githubusercontent.com/udacity/deep-learning/master/first-neural-network/Bike-Sharing-Dataset/hour.csv', download_timeout_seconds=30, regime_quantiles=(0.33, 0.67), chronological_train_fraction=0.8, high_episode_quantile=0.9, brownian_mc_paths=5000, brownian_mc_steps=500)

# 2. Data-source Strategy pattern

`UCIHourlyBikeSource` downloads and caches the official ZIP. If you already have `hour.csv`, switch to `LocalCSVSource`.

The rest of the notebook only calls `.load()`.

In [3]:
class DatasetSource(ABC):
    @abstractmethod
    def load(self) -> pd.DataFrame:
        raise NotImplementedError


class UCIHourlyBikeSource(DatasetSource):
    """Load UCI hour.csv with a local cache and a read-only mirror fallback.

    Resolution order:
    1. Existing local cache ``cache_dir/hour.csv``.
    2. Official UCI ZIP.
    3. Public GitHub mirror of the same UCI ``hour.csv``.

    The fallback is deliberately only a resilience mechanism; the official UCI
    repository remains the primary source.
    """

    def __init__(
        self,
        url: str,
        cache_dir: str = "./data",
        fallback_csv_url: Optional[str] = None,
        timeout_seconds: int = 30,
    ):
        self.url = url
        self.fallback_csv_url = fallback_csv_url
        self.timeout_seconds = int(timeout_seconds)
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.cached_csv = self.cache_dir / "hour.csv"

    @staticmethod
    def _request_bytes(url: str, timeout_seconds: int) -> bytes:
        request = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0 ST5221-notebook"},
        )
        with urllib.request.urlopen(request, timeout=timeout_seconds) as response:
            return response.read()

    def _try_official_zip(self) -> None:
        payload = self._request_bytes(self.url, self.timeout_seconds)
        with zipfile.ZipFile(io.BytesIO(payload)) as zf:
            candidates = [name for name in zf.namelist() if name.endswith("hour.csv")]
            if not candidates:
                raise FileNotFoundError("The downloaded UCI ZIP does not contain hour.csv.")
            with zf.open(candidates[0]) as src:
                self.cached_csv.write_bytes(src.read())

    def _try_fallback_csv(self) -> None:
        if not self.fallback_csv_url:
            raise RuntimeError("No fallback CSV URL configured.")
        payload = self._request_bytes(self.fallback_csv_url, self.timeout_seconds)
        self.cached_csv.write_bytes(payload)

    def _download_if_needed(self) -> None:
        if self.cached_csv.exists():
            return

        errors: list[str] = []

        try:
            print("Downloading official UCI Bike Sharing ZIP...")
            self._try_official_zip()
            return
        except Exception as exc:
            errors.append(f"official UCI source: {type(exc).__name__}: {exc}")

        try:
            print("Official source unavailable; trying public hour.csv mirror...")
            self._try_fallback_csv()
            return
        except Exception as exc:
            errors.append(f"fallback mirror: {type(exc).__name__}: {exc}")

        detail = "\n  - ".join(errors)
        raise RuntimeError(
            "Could not obtain hour.csv automatically.\n"
            "Download hour.csv manually from the UCI Bike Sharing Dataset and "
            f"place it at: {self.cached_csv.resolve()}\n"
            f"Attempt details:\n  - {detail}"
        )

    def load(self) -> pd.DataFrame:
        self._download_if_needed()
        return pd.read_csv(self.cached_csv)


class LocalCSVSource(DatasetSource):
    def __init__(self, csv_path: str | Path):
        self.csv_path = Path(csv_path)

    def load(self) -> pd.DataFrame:
        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV not found: {self.csv_path.resolve()}")
        return pd.read_csv(self.csv_path)


def prepare_hourly_frame(raw: pd.DataFrame) -> pd.DataFrame:
    required = {
        "dteday", "hr", "cnt", "casual", "registered",
        "workingday", "weekday", "weathersit",
        "temp", "hum", "windspeed", "season", "mnth",
    }
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = raw.copy()
    df["timestamp"] = pd.to_datetime(df["dteday"], errors="raise") + pd.to_timedelta(df["hr"], unit="h")
    df = df.sort_values("timestamp").reset_index(drop=True)

    if df["timestamp"].duplicated().any():
        raise ValueError("Duplicate hourly timestamps detected; inspect the input dataset.")

    # Stable category vocabularies prevent Patsy/statsmodels train/test
    # prediction failures when a chronological split omits a category level.
    category_levels = {
        "hr": list(range(24)),
        "weekday": list(range(7)),
        "weathersit": [1, 2, 3, 4],
        "season": [1, 2, 3, 4],
        "mnth": list(range(1, 13)),
    }
    for col, levels in category_levels.items():
        unknown = set(pd.Series(df[col]).dropna().unique()) - set(levels)
        if unknown:
            raise ValueError(f"Unexpected values in {col}: {sorted(unknown)}")
        df[col] = pd.Categorical(df[col], categories=levels)

    numeric_cols = ["cnt", "casual", "registered", "workingday", "temp", "hum", "windspeed"]
    if df[numeric_cols].isna().any().any():
        bad = df[numeric_cols].columns[df[numeric_cols].isna().any()].tolist()
        raise ValueError(f"Missing values detected in required numeric columns: {bad}")

    if (df[["cnt", "casual", "registered"]] < 0).any().any():
        raise ValueError("Rental counts must be non-negative.")

    # Time features useful for later stochastic diagnostics.
    df["hour"] = df["timestamp"].dt.hour
    df["date"] = df["timestamp"].dt.date
    df["is_consecutive"] = df["timestamp"].diff().eq(pd.Timedelta(hours=1))

    return df


source: DatasetSource = UCIHourlyBikeSource(
    url=CFG.dataset_url,
    cache_dir=CFG.cache_dir,
    fallback_csv_url=CFG.dataset_fallback_csv_url,
    timeout_seconds=CFG.download_timeout_seconds,
)

df = prepare_hourly_frame(source.load())

print(f"Rows: {len(df):,}")
print(f"Time span: {df['timestamp'].min()} to {df['timestamp'].max()}")
display(df.head())

Rows: 17,379
Time span: 2011-01-01 00:00:00 to 2012-12-31 23:00:00


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,timestamp,hour,date,is_consecutive
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00,0,2011-01-01,False
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00,1,2011-01-01,True
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00,2,2011-01-01,True
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00,3,2011-01-01,True
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00,4,2011-01-01,True


> **Offline fallback:** if the UCI URL is inaccessible in your environment, download `hour.csv` from the official dataset page and replace the source construction with:
>
> `source = LocalCSVSource("path/to/hour.csv")`

---

# 3. Reusable Bokeh plotting façade

A small façade keeps plots concise and avoids repeated Bokeh boilerplate.

`safe_table` also normalizes dataframe column names before creating a `ColumnDataSource`; this avoids a common Bokeh issue when dataframe columns are integers or mixed types.

In [4]:
class PlotFactory:
    @staticmethod
    def _is_datetime_like(values) -> bool:
        arr = np.asarray(values)
        return pd.api.types.is_datetime64_any_dtype(arr.dtype)

    @staticmethod
    def line(
        x,
        y,
        title: str,
        x_label: str,
        y_label: str,
        width: int = 900,
        height: int = 350,
    ):
        p = figure(
            width=width,
            height=height,
            title=title,
            x_axis_type="datetime" if PlotFactory._is_datetime_like(x) else "linear",
            tools="pan,wheel_zoom,box_zoom,reset,save",
        )
        p.line(x, y, line_width=2)
        p.xaxis.axis_label = x_label
        p.yaxis.axis_label = y_label
        return p

    @staticmethod
    def multi_line(
        x,
        series: Dict[str, Sequence[float]],
        title: str,
        x_label: str,
        y_label: str,
        width: int = 900,
        height: int = 380,
    ):
        p = figure(
            width=width,
            height=height,
            title=title,
            x_axis_type="datetime" if PlotFactory._is_datetime_like(x) else "linear",
            tools="pan,wheel_zoom,box_zoom,reset,save",
        )
        for label, values in series.items():
            p.line(x, values, line_width=2, legend_label=label)
        if p.legend:
            p.legend.location = "top_right"
        p.xaxis.axis_label = x_label
        p.yaxis.axis_label = y_label
        return p

    @staticmethod
    def heatmap(
        matrix: np.ndarray,
        labels: Sequence[str],
        title: str,
        value_name: str = "value",
        width: int = 620,
        height: int = 520,
    ):
        matrix = np.asarray(matrix, dtype=float)
        if matrix.shape != (len(labels), len(labels)):
            raise ValueError(
                f"Expected a {len(labels)}x{len(labels)} matrix; got {matrix.shape}."
            )
        if not np.isfinite(matrix).any():
            raise ValueError("Heatmap matrix contains no finite values.")

        rows = []
        for i, src in enumerate(labels):
            for j, dst in enumerate(labels):
                rows.append({"source": src, "target": dst, value_name: matrix[i, j]})

        long = pd.DataFrame(rows)
        finite = matrix[np.isfinite(matrix)]
        low, high = float(finite.min()), float(finite.max())
        if np.isclose(low, high):
            high = low + 1e-12

        mapper = LinearColorMapper(palette=Viridis256, low=low, high=high)

        p = figure(
            x_range=list(labels),
            y_range=list(reversed(labels)),
            width=width,
            height=height,
            title=title,
            toolbar_location=None,
        )
        p.rect(
            x="target",
            y="source",
            width=1,
            height=1,
            source=ColumnDataSource(long),
            fill_color={"field": value_name, "transform": mapper},
            line_color="white",
        )
        p.add_tools(HoverTool(
            tooltips=[
                ("from", "@source"),
                ("to", "@target"),
                (value_name, f"@{value_name}{{0.0000}}"),
            ]
        ))
        p.add_layout(ColorBar(color_mapper=mapper), "right")
        p.xaxis.axis_label = "Next state"
        p.yaxis.axis_label = "Current state"
        return p

    @staticmethod
    def histogram_with_pdfs(
        values: Sequence[float],
        pdfs: Dict[str, Callable[[np.ndarray], np.ndarray]],
        title: str,
        x_label: str,
        bins: int = 30,
        width: int = 900,
        height: int = 380,
    ):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            raise ValueError("No finite observations supplied to histogram.")

        hist, edges = np.histogram(values, bins=bins, density=True)

        p = figure(
            width=width,
            height=height,
            title=title,
            tools="pan,wheel_zoom,box_zoom,reset,save",
        )
        p.quad(
            top=hist,
            bottom=0,
            left=edges[:-1],
            right=edges[1:],
            fill_alpha=0.35,
            line_alpha=0.4,
            legend_label="Empirical",
        )

        grid = np.linspace(max(1e-9, values.min()), values.max(), 400)
        for label, pdf in pdfs.items():
            p.line(grid, pdf(grid), line_width=2, legend_label=label)

        if p.legend:
            p.legend.location = "top_right"
        p.xaxis.axis_label = x_label
        p.yaxis.axis_label = "Density"
        return p

    @staticmethod
    def safe_table(
        frame: pd.DataFrame,
        title: str,
        width: int = 900,
        height: int = 250,
    ):
        data = frame.copy()
        data.columns = [str(c) for c in data.columns]
        data = data.reset_index(drop=True)

        for col in data.columns:
            if data[col].dtype == "object":
                data[col] = data[col].astype(str)

        source = ColumnDataSource(data)
        columns = [
            TableColumn(field=c, title=c.replace("_", " ").title())
            for c in data.columns
        ]
        table = DataTable(
            source=source,
            columns=columns,
            width=width,
            height=height,
            index_position=None,
        )
        title_div = Div(text=f"<h3 style='margin:0 0 8px 0'>{title}</h3>", width=width)
        return column(title_div, table)

In [26]:
summary = pd.DataFrame({
    "metric": [
        "rows",
        "start",
        "end",
        "mean hourly rentals",
        "variance hourly rentals",
        "variance / mean",
        "fraction consecutive hourly rows",
    ],
    "value": [
        len(df),
        df["timestamp"].min(),
        df["timestamp"].max(),
        df["cnt"].mean(),
        df["cnt"].var(ddof=1),
        df["cnt"].var(ddof=1) / df["cnt"].mean(),
        df["is_consecutive"].mean(),
    ],
})

display(summary)
show(PlotFactory.safe_table(summary, "Dataset sanity check"))

sample = df.iloc[:24 * 14]
show(
    PlotFactory.line(
        sample["timestamp"].to_numpy(),
        sample["cnt"].to_numpy(),
        title="Two weeks of observed hourly bike rentals",
        x_label="Time",
        y_label="Hourly rental count",
    )
)

,metric,value
0,rows,17379
1,start,2011-01-01 00:00:00
2,end,2012-12-31 23:00:00
3,mean hourly rentals,189.463088
4,variance hourly rentals,32901.461104
5,variance / mean,173.656312
6,fraction consecutive hourly rows,0.995627


## Interpretation

A stochastic model is always an abstraction.

The bike counts contain:

- deterministic seasonality;
- working-day effects;
- weather effects;
- autocorrelation;
- random shocks.

A good stochastic analysis asks:

$$
\text{Which part is structural and predictable, and which part is stochastic?}
$$

We will therefore use different state representations for different questions rather than assuming one universal process.

---

# 4. Demand-regime Strategy

A Markov chain needs a state space. Raw counts are too granular for an introductory finite-state chain, so we construct three regimes:

$$
S=\{\text{Low},\text{Medium},\text{High}\}.
$$

The thresholds are learned **only from the training period** to avoid leaking future distributional information into the state construction.

In [6]:
class DemandRegimeStrategy(ABC):
    @abstractmethod
    def fit(self, y: pd.Series) -> "DemandRegimeStrategy":
        raise NotImplementedError

    @abstractmethod
    def transform(self, y: pd.Series) -> pd.Categorical:
        raise NotImplementedError


class QuantileRegimeStrategy(DemandRegimeStrategy):
    labels = ("Low", "Medium", "High")

    def __init__(self, quantiles=(0.33, 0.67)):
        self.quantiles = quantiles
        self.thresholds_: Optional[np.ndarray] = None

    def fit(self, y: pd.Series) -> "QuantileRegimeStrategy":
        self.thresholds_ = y.quantile(self.quantiles).to_numpy(dtype=float)
        return self

    def transform(self, y: pd.Series) -> pd.Categorical:
        if self.thresholds_ is None:
            raise RuntimeError("Call fit() before transform().")

        lo, hi = self.thresholds_
        values = np.select(
            [y <= lo, y <= hi],
            ["Low", "Medium"],
            default="High",
        )
        return pd.Categorical(
            values,
            categories=list(self.labels),
            ordered=True,
        )


split_idx = int(len(df) * CFG.chronological_train_fraction)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()

regime_strategy = QuantileRegimeStrategy(CFG.regime_quantiles).fit(train["cnt"])

df["regime"] = regime_strategy.transform(df["cnt"])
train["regime"] = regime_strategy.transform(train["cnt"])
test["regime"] = regime_strategy.transform(test["cnt"])

print("Training thresholds:", regime_strategy.thresholds_)
display(df[["timestamp", "cnt", "regime"]].head(10))

Training thresholds: [ 64. 209.]


,timestamp,cnt,regime
0,2011-01-01 00:00:00,16,Low
1,2011-01-01 01:00:00,40,Low
2,2011-01-01 02:00:00,32,Low
3,2011-01-01 03:00:00,13,Low
4,2011-01-01 04:00:00,1,Low
5,2011-01-01 05:00:00,1,Low
6,2011-01-01 06:00:00,2,Low
7,2011-01-01 07:00:00,3,Low
8,2011-01-01 08:00:00,8,Low
9,2011-01-01 09:00:00,14,Low


# 5. Discrete-time Markov chain estimator

For a time-homogeneous chain,

$$
p_{ij}
=
P(X_{t+1}=j\mid X_t=i).
$$

The maximum-likelihood estimator from observed transition counts is

$$
\hat p_{ij}
=
\frac{N_{ij}}
{\sum_k N_{ik}},
$$

where $N_{ij}$ counts observed transitions from $i$ to $j$.

We only use pairs of rows that are exactly one hour apart.

The class below also implements:

- stationary distribution;
- power iteration;
- spectral gap;
- communicating classes;
- irreducibility;
- aperiodicity;
- $n$-step transition matrices;
- mean hitting times.

In [7]:
class FiniteMarkovChain:
    def __init__(self, state_labels: Sequence[str]):
        self.state_labels = tuple(state_labels)
        self.state_to_idx = {s: i for i, s in enumerate(self.state_labels)}
        self.counts_: Optional[np.ndarray] = None
        self.P_: Optional[np.ndarray] = None

    def fit(
        self,
        states: Sequence[str],
        timestamps: Optional[Sequence[pd.Timestamp]] = None,
    ) -> "FiniteMarkovChain":
        states = np.asarray(states, dtype=object)
        if len(states) < 2:
            raise ValueError("At least two observations are required to estimate transitions.")
        unknown = sorted(set(map(str, states)) - set(self.state_labels))
        if unknown:
            raise ValueError(f"Unknown state labels encountered: {unknown}")
        m = len(self.state_labels)
        counts = np.zeros((m, m), dtype=int)

        if timestamps is None:
            valid_pair = np.ones(len(states) - 1, dtype=bool)
        else:
            ts = pd.to_datetime(np.asarray(timestamps))
            valid_pair = np.diff(ts) == np.timedelta64(1, "h")

        for k in range(len(states) - 1):
            if not valid_pair[k]:
                continue
            i = self.state_to_idx[str(states[k])]
            j = self.state_to_idx[str(states[k + 1])]
            counts[i, j] += 1

        row_sums = counts.sum(axis=1, keepdims=True)
        if np.any(row_sums == 0):
            raise ValueError("At least one state has no outgoing observed transitions.")

        self.counts_ = counts
        self.P_ = counts / row_sums
        return self

    @property
    def P(self) -> np.ndarray:
        if self.P_ is None:
            raise RuntimeError("Fit the chain first.")
        return self.P_

    def n_step(self, n: int) -> np.ndarray:
        return np.linalg.matrix_power(self.P, n)

    def stationary_eigen(self) -> np.ndarray:
        values, vectors = np.linalg.eig(self.P.T)
        idx = int(np.argmin(np.abs(values - 1.0)))
        v = np.real_if_close(vectors[:, idx]).astype(float)
        # Eigenvectors are defined only up to sign. Flip instead of taking
        # elementwise absolute values, which can hide numerical/model issues.
        if v.sum() < 0:
            v = -v
        v[np.abs(v) < 1e-14] = 0.0
        if (v < -1e-10).any() or np.isclose(v.sum(), 0.0):
            raise RuntimeError("Could not recover a valid stationary probability vector.")
        v = np.clip(v, 0.0, None)
        return v / v.sum()

    def stationary_power(
        self,
        tol: float = 1e-13,
        max_iter: int = 100_000,
    ) -> np.ndarray:
        pi = np.full(len(self.state_labels), 1.0 / len(self.state_labels))

        for _ in range(max_iter):
            nxt = pi @ self.P
            if np.linalg.norm(nxt - pi, ord=1) < tol:
                return nxt / nxt.sum()
            pi = nxt

        raise RuntimeError("Power iteration did not converge.")

    def spectral_gap(self) -> float:
        if len(self.state_labels) < 2:
            return float("nan")
        eigvals = np.linalg.eigvals(self.P)
        moduli = np.sort(np.abs(eigvals))[::-1]
        return float(max(0.0, 1.0 - moduli[1]))

    def graph(self) -> nx.DiGraph:
        g = nx.DiGraph()
        g.add_nodes_from(self.state_labels)
        for i, src in enumerate(self.state_labels):
            for j, dst in enumerate(self.state_labels):
                if self.P[i, j] > 0:
                    g.add_edge(src, dst, weight=float(self.P[i, j]))
        return g

    def communicating_classes(self) -> list[set[str]]:
        return [set(c) for c in nx.strongly_connected_components(self.graph())]

    def is_irreducible(self) -> bool:
        return nx.is_strongly_connected(self.graph())

    def is_aperiodic(self) -> bool:
        if not self.is_irreducible():
            return False
        return nx.is_aperiodic(self.graph())

    def mean_hitting_times(self, target_state: str) -> pd.Series:
        target = self.state_to_idx[target_state]
        non_target = [i for i in range(len(self.state_labels)) if i != target]

        Q = self.P[np.ix_(non_target, non_target)]
        # m = 1 + Qm  =>  (I-Q)m = 1
        m = np.linalg.solve(np.eye(len(non_target)) - Q, np.ones(len(non_target)))

        result = np.zeros(len(self.state_labels), dtype=float)
        result[non_target] = m
        result[target] = 0.0

        return pd.Series(result, index=self.state_labels, name=f"E[T_{target_state}]")


markov = FiniteMarkovChain(regime_strategy.labels).fit(
    train["regime"].astype(str),
    train["timestamp"],
)

display(pd.DataFrame(
    markov.counts_,
    index=markov.state_labels,
    columns=markov.state_labels,
))
display(pd.DataFrame(
    markov.P,
    index=markov.state_labels,
    columns=markov.state_labels,
).round(4))

,Low,Medium,High
Low,3809,709,6
Medium,713,3114,914
High,2,917,3648


,Low,Medium,High
Low,0.8420,0.1567,0.0013
Medium,0.1504,0.6568,0.1928
High,0.0004,0.2008,0.7988


In [27]:
show(
    PlotFactory.heatmap(
        markov.P,
        labels=markov.state_labels,
        title="Estimated one-hour Markov transition matrix",
        value_name="probability",
    )
)

pi_eigen = markov.stationary_eigen()
pi_power = markov.stationary_power()

markov_diagnostics = pd.DataFrame({
    "quantity": [
        "irreducible",
        "aperiodic",
        "communicating classes",
        "spectral gap",
        "stationary distribution: Low",
        "stationary distribution: Medium",
        "stationary distribution: High",
        "max |eigen - power|",
    ],
    "value": [
        markov.is_irreducible(),
        markov.is_aperiodic(),
        markov.communicating_classes(),
        markov.spectral_gap(),
        pi_eigen[0],
        pi_eigen[1],
        pi_eigen[2],
        np.max(np.abs(pi_eigen - pi_power)),
    ],
})

display(markov_diagnostics)

,quantity,value
0,irreducible,True
1,aperiodic,True
2,communicating classes,"[{High, Medium, Low}]"
3,spectral gap,0.176465
4,stationary distribution: Low,0.326951
5,stationary distribution: Medium,0.342632
6,stationary distribution: High,0.330417
7,max |eigen - power|,0.0


## How to read the transition matrix

A large diagonal element $\hat p_{ii}$ means a demand regime is **persistent**.

For example,

$$
\hat p_{\text{High,High}}
$$

estimates the probability that a High-demand hour is followed by another High-demand hour.

An off-diagonal probability such as

$$
\hat p_{\text{Medium,High}}
$$

describes regime escalation.

### Stationarity

A stationary probability vector satisfies

$$
\pi P=\pi,
\qquad
\sum_i\pi_i=1.
$$

It is therefore a left eigenvector of $P$ with eigenvalue $1$.

If the chain is finite, irreducible and aperiodic, then

$$
P^n(i,j)\rightarrow \pi_j.
$$

The initial state is gradually forgotten.

---

# 6. Empirical mixing and the spectral-gap idea

For each possible initial state, compute

$$
\mu_n=e_iP^n
$$

and compare it with $\pi$ using total-variation distance:

$$
d_{\text{TV}}(\mu,\pi)
=
\frac12\sum_j|\mu_j-\pi_j|.
$$

A larger spectral gap usually indicates faster convergence.

In [28]:
def total_variation(p: np.ndarray, q: np.ndarray) -> float:
    return 0.5 * np.abs(np.asarray(p) - np.asarray(q)).sum()


def mixing_curve(
    chain: FiniteMarkovChain,
    max_steps: int = 48,
) -> pd.DataFrame:
    pi = chain.stationary_eigen()
    rows = []

    for i, label in enumerate(chain.state_labels):
        mu = np.zeros(len(chain.state_labels))
        mu[i] = 1.0

        for n in range(max_steps + 1):
            rows.append({
                "initial_state": label,
                "step": n,
                "tv_distance": total_variation(mu, pi),
            })
            mu = mu @ chain.P

    return pd.DataFrame(rows)


mix = mixing_curve(markov, max_steps=48)

series = {
    state: mix.loc[mix["initial_state"] == state, "tv_distance"].to_numpy()
    for state in markov.state_labels
}

show(
    PlotFactory.multi_line(
        x=np.arange(49),
        series=series,
        title="Convergence toward the stationary distribution",
        x_label="Hours / Markov steps",
        y_label="Total-variation distance to stationary distribution",
    )
)

# 7. First-passage / hitting-time analysis

Let

$$
T_H=\inf\{n\ge 0:X_n=\text{High}\}.
$$

For non-target states,

$$
m_i
=
E_i[T_H]
=
1+\sum_jp_{ij}m_j.
$$

After removing the target state this becomes a linear system:

$$
(I-Q)m=\mathbf 1.
$$

This is a recurring ST5221 technique:

$$
\boxed{
\text{condition on the first step}
\rightarrow
\text{recursion}
\rightarrow
\text{linear algebra}
}
$$

In [29]:
hitting_high = markov.mean_hitting_times("High").reset_index()
hitting_high.columns = ["starting_state", "expected_hours_to_high"]

display(hitting_high.round(3))
show(PlotFactory.safe_table(
    hitting_high.round(3),
    "Mean first-passage time to High demand",
    height=180,
))

,starting_state,expected_hours_to_high
0,Low,16.300
1,Medium,10.057
2,High,0.000


### Complexity

For $m$ finite states:

- counting transitions is $\Theta(n)$ in the observed sequence length;
- dense matrix multiplication is $\Theta(m^3)$ in the classical implementation for arbitrary matrix powers, although repeated-vector propagation is cheaper;
- solving dense hitting-time systems is $O(m^3)$;
- sparse chains can exploit sparse linear algebra and graph algorithms.

For our three-state example the cost is trivial, but the same mathematics scales to much larger state spaces.

---

# 8. Continuous-time Markov-chain approximation

A CTMC is governed by a generator matrix $Q$.

For $i\neq j$,

$$
q_{ij}
$$

is an instantaneous transition rate, while

$$
q_{ii}
=
-\sum_{j\neq i}q_{ij}.
$$

For a fully observed continuous-time path, the likelihood estimator is

$$
\hat q_{ij}
=
\frac{N_{ij}}{T_i},
$$

where $T_i$ is total exposure time spent in state $i$.

Then

$$
P(t)=e^{Qt}.
$$

### Important data limitation

Our data are hourly snapshots. We do **not** observe any transitions that happen and reverse within an hour.

Therefore the generator below is an **hourly panel approximation**, useful for learning the theory but not a claim that the true Capital Bikeshare demand process is continuously observed.

In [30]:
class CTMCFromSnapshots:
    def __init__(self, state_labels: Sequence[str]):
        self.state_labels = tuple(state_labels)
        self.state_to_idx = {s: i for i, s in enumerate(self.state_labels)}
        self.Q_: Optional[np.ndarray] = None
        self.exposure_: Optional[np.ndarray] = None
        self.jump_counts_: Optional[np.ndarray] = None

    def fit(
        self,
        states: Sequence[str],
        timestamps: Sequence[pd.Timestamp],
    ) -> "CTMCFromSnapshots":
        states = np.asarray(states, dtype=object)
        ts = pd.to_datetime(np.asarray(timestamps))
        m = len(self.state_labels)

        jumps = np.zeros((m, m), dtype=int)
        exposure = np.zeros(m, dtype=float)

        for k in range(len(states) - 1):
            delta_h = (ts[k + 1] - ts[k]) / np.timedelta64(1, "h")
            if not np.isclose(delta_h, 1.0):
                continue

            i = self.state_to_idx[str(states[k])]
            j = self.state_to_idx[str(states[k + 1])]

            exposure[i] += float(delta_h)
            if i != j:
                jumps[i, j] += 1

        Q = np.zeros((m, m), dtype=float)

        for i in range(m):
            if exposure[i] <= 0:
                raise ValueError(f"No exposure for state {self.state_labels[i]}")

            for j in range(m):
                if i != j:
                    Q[i, j] = jumps[i, j] / exposure[i]

            Q[i, i] = -Q[i].sum()

        self.Q_ = Q
        self.exposure_ = exposure
        self.jump_counts_ = jumps
        return self

    @property
    def Q(self) -> np.ndarray:
        if self.Q_ is None:
            raise RuntimeError("Fit the CTMC first.")
        return self.Q_

    def transition_matrix(self, t: float) -> np.ndarray:
        return linalg.expm(self.Q * t)

    def stationary_distribution(self) -> np.ndarray:
        # Solve Q^T pi = 0 with the normalization sum(pi)=1.
        A = self.Q.T.copy()
        b = np.zeros(len(self.state_labels))

        A[-1, :] = 1.0
        b[-1] = 1.0

        pi = np.linalg.solve(A, b)
        return pi

    def mean_holding_times(self) -> pd.Series:
        rates = -np.diag(self.Q)
        values = np.where(rates > 0, 1.0 / rates, np.inf)
        return pd.Series(values, index=self.state_labels, name="mean_holding_hours")


ctmc = CTMCFromSnapshots(regime_strategy.labels).fit(
    train["regime"].astype(str),
    train["timestamp"],
)

Q = ctmc.Q
P_from_Q_1h = ctmc.transition_matrix(1.0)

display(pd.DataFrame(Q, index=ctmc.state_labels, columns=ctmc.state_labels).round(4))
display(ctmc.mean_holding_times().round(3))

,Low,Medium,High
Low,-0.1580,0.1567,0.0013
Medium,0.1504,-0.3432,0.1928
High,0.0004,0.2008,-0.2012


Low       6.327
Medium    2.914
High      4.970
Name: mean_holding_hours, dtype: float64

In [31]:
comparison_rows = []
for i, src in enumerate(markov.state_labels):
    for j, dst in enumerate(markov.state_labels):
        comparison_rows.append({
            "from": src,
            "to": dst,
            "empirical_DTMC_P": markov.P[i, j],
            "CTMC_expm_Q_1h": P_from_Q_1h[i, j],
            "absolute_difference": abs(markov.P[i, j] - P_from_Q_1h[i, j]),
        })

ctmc_comparison = pd.DataFrame(comparison_rows)

display(ctmc_comparison.round(4))

show(
    PlotFactory.heatmap(
        P_from_Q_1h,
        labels=ctmc.state_labels,
        title=r"CTMC-implied one-hour transition matrix exp(Q)",
        value_name="probability",
    )
)

print("CTMC stationary distribution:")
display(pd.Series(
    ctmc.stationary_distribution(),
    index=ctmc.state_labels,
    name="pi",
).round(4))

,from,to,empirical_DTMC_P,CTMC_expm_Q_1h,absolute_difference
0,Low,Low,0.8420,0.8633,0.0214
1,Low,Medium,0.1567,0.1235,0.0332
2,Low,High,0.0013,0.0131,0.0118
3,Medium,Low,0.1504,0.1185,0.0319
4,Medium,Medium,0.6568,0.7330,0.0761
5,Medium,High,0.1928,0.1486,0.0442
6,High,Low,0.0004,0.0124,0.0119
7,High,Medium,0.2008,0.1547,0.0461
8,High,High,0.7988,0.8329,0.0341


CTMC stationary distribution:


Low       0.3270
Medium    0.3426
High      0.3304
Name: pi, dtype: float64

## Why $e^{Qt}$ appears

The CTMC Chapman–Kolmogorov relation is

$$
P(t+s)=P(t)P(s).
$$

At infinitesimal scale,

$$
P(h)=I+Qh+o(h).
$$

Differentiation gives the Kolmogorov equation

$$
\frac{dP(t)}{dt}=P(t)Q.
$$

With $P(0)=I$, its matrix solution is

$$
\boxed{
P(t)=e^{Qt}
}
$$

where

$$
e^{Qt}
=
I+Qt+\frac{Q^2t^2}{2!}+\cdots.
$$

This parallels the discrete-time relation $P^n$.

---

# 9. Poisson count modelling

A homogeneous Poisson process would imply that counts over equal-length disjoint windows are approximately:

$$
N(t+h)-N(t)
\sim
\operatorname{Poisson}(\lambda h).
$$

For a Poisson random variable,

$$
E[N]=\operatorname{Var}(N)=\lambda.
$$

Bike demand plainly varies by hour, weather and working-day status, so a **single homogeneous $\lambda$ is not realistic**.

A more useful approximation is a piecewise non-homogeneous count model:

$$
Y_t\mid x_t
\sim
\operatorname{Poisson}(\lambda_t),
$$

$$
\log \lambda_t
=
\beta_0+x_t^\top\beta.
$$

We fit a Poisson GLM to model the predictable intensity.

In [32]:
class PoissonDemandModel:
    def __init__(self, formula: str):
        self.formula = formula
        self.result_ = None

    def fit(self, frame: pd.DataFrame) -> "PoissonDemandModel":
        self.result_ = smf.glm(
            formula=self.formula,
            data=frame,
            family=sm.families.Poisson(),
        ).fit()
        return self

    def predict_rate(self, frame: pd.DataFrame) -> np.ndarray:
        if self.result_ is None:
            raise RuntimeError("Fit the model first.")
        return np.asarray(self.result_.predict(frame), dtype=float)

    def pearson_dispersion(self) -> float:
        if self.result_ is None:
            raise RuntimeError("Fit the model first.")
        return float(self.result_.pearson_chi2 / self.result_.df_resid)


poisson_formula = (
    "cnt ~ C(hr) + C(weekday) + C(weathersit) "
    "+ workingday + temp + hum + windspeed + C(season) + C(mnth)"
)

poisson_model = PoissonDemandModel(poisson_formula).fit(train)

test["lambda_hat"] = poisson_model.predict_rate(test)

poisson_metrics = pd.DataFrame({
    "metric": [
        "MAE",
        "RMSE",
        "mean Poisson deviance",
        "Pearson dispersion",
        "raw count variance / mean",
    ],
    "value": [
        mean_absolute_error(test["cnt"], test["lambda_hat"]),
        np.sqrt(mean_squared_error(test["cnt"], test["lambda_hat"])),
        mean_poisson_deviance(test["cnt"], np.clip(test["lambda_hat"], 1e-9, None)),
        poisson_model.pearson_dispersion(),
        train["cnt"].var(ddof=1) / train["cnt"].mean(),
    ],
})

display(poisson_metrics.round(4))

,metric,value
0,MAE,104.6033
1,RMSE,156.5634
2,mean Poisson deviance,81.4674
3,Pearson dispersion,38.9535
4,raw count variance / mean,159.6098


## Overdispersion diagnostic

For an ideal Poisson model,

$$
\operatorname{Var}(Y_t\mid X_t)
\approx
E(Y_t\mid X_t).
$$

The Pearson dispersion statistic is approximately

$$
\hat\phi
=
\frac{\sum_t r_{P,t}^2}{\text{residual degrees of freedom}},
$$

where

$$
r_{P,t}
=
\frac{y_t-\hat\lambda_t}
{\sqrt{\hat\lambda_t}}.
$$

Interpretation:

- $\hat\phi\approx1$: Poisson variance assumption is plausible;
- $\hat\phi\gg1$: overdispersion;
- $\hat\phi<1$: underdispersion.

Overdispersion can arise because a simple Poisson model does not capture all latent heterogeneity, clustering or dependence.

In [33]:
hour_profile = (
    test.groupby("hr", as_index=False, observed=True)
        .agg(
            observed_mean=("cnt", "mean"),
            predicted_mean=("lambda_hat", "mean"),
        )
)

show(
    PlotFactory.multi_line(
        x=hour_profile["hr"].to_numpy(),
        series={
            "Observed hourly mean": hour_profile["observed_mean"].to_numpy(),
            "Poisson predicted mean": hour_profile["predicted_mean"].to_numpy(),
        },
        title="Observed versus Poisson-GLM mean demand by hour of day",
        x_label="Hour of day",
        y_label="Mean rentals",
    )
)

display(hour_profile.round(2))

,hr,observed_mean,predicted_mean
0,0,69.29,47.11
1,1,43.60,29.10
2,2,28.28,20.43
3,3,14.13,10.56
4,4,8.63,5.50
5,5,28.60,17.07
6,6,101.22,66.41
7,7,286.36,185.10
8,8,485.34,313.64
9,9,291.43,193.20


# 10. Poisson simulation: model-based stochastic uncertainty

Conditional on estimated rates $\hat\lambda_t$, simulate

$$
Y_t^{(b)}
\sim
\operatorname{Poisson}(\hat\lambda_t).
$$

This distinguishes:

- **mean prediction:** $\hat\lambda_t$;
- **one possible stochastic realization:** $Y_t^{(b)}$.

A stochastic model is not merely a predictor. It specifies a **distribution over possible future paths**.

In [34]:
class PiecewisePoissonSimulator:
    def __init__(self, rng: np.random.Generator):
        self.rng = rng

    def simulate_counts(self, rates: Sequence[float]) -> np.ndarray:
        rates = np.clip(np.asarray(rates, dtype=float), 1e-12, None)
        return self.rng.poisson(rates)


simulator = PiecewisePoissonSimulator(np.random.default_rng(CFG.random_seed))

comparison = test.iloc[:24 * 7].copy()
comparison["simulated_poisson"] = simulator.simulate_counts(comparison["lambda_hat"])

p = figure(
    width=950,
    height=400,
    x_axis_type="datetime",
    title="One observed week versus one Poisson-GLM stochastic realization",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p.line(
    comparison["timestamp"],
    comparison["cnt"],
    line_width=2,
    legend_label="Observed",
)
p.line(
    comparison["timestamp"],
    comparison["lambda_hat"],
    line_width=2,
    legend_label="Estimated intensity",
)
p.scatter(
    comparison["timestamp"],
    comparison["simulated_poisson"],
    size=5,
    alpha=0.6,
    legend_label="Simulated count",
)
p.legend.location = "top_left"
p.xaxis.axis_label = "Time"
p.yaxis.axis_label = "Hourly count"
show(p)

# 11. Renewal-process view: High-demand episode starts

A renewal process has IID inter-renewal times

$$
X_1,X_2,\ldots
$$

and renewal epochs

$$
S_n=X_1+\cdots+X_n.
$$

The associated counting process is

$$
N(t)=\max\{n:S_n\le t\}.
$$

To construct an empirical event sequence, define a **High-demand episode** as a consecutive run of hours above a high threshold.

The event time is the start of the run.

This gives us observed inter-event gaps.

### Why this is interesting

If episode starts formed a homogeneous Poisson process, the inter-event gaps would be exponential and would have coefficient of variation

$$
CV=\frac{\sigma}{\mu}=1.
$$

If the gaps instead cluster around daily cycles, the exponential assumption should fail.

That failure is itself informative.

In [35]:
@dataclass
class Episode:
    start: pd.Timestamp
    end: pd.Timestamp
    duration_hours: int
    reward_rentals: int


class HighDemandEpisodeExtractor:
    def __init__(self, threshold: float):
        self.threshold = float(threshold)

    def extract(self, frame: pd.DataFrame) -> list[Episode]:
        x = frame.sort_values("timestamp").reset_index(drop=True).copy()
        x["is_high_event"] = x["cnt"] >= self.threshold

        episodes: list[Episode] = []
        start_idx: Optional[int] = None

        for i in range(len(x)):
            active = bool(x.loc[i, "is_high_event"])

            if active and start_idx is None:
                start_idx = i

            next_is_contiguous_high = False
            if active and i + 1 < len(x):
                next_is_contiguous_high = (
                    bool(x.loc[i + 1, "is_high_event"])
                    and (x.loc[i + 1, "timestamp"] - x.loc[i, "timestamp"])
                    == pd.Timedelta(hours=1)
                )

            if active and not next_is_contiguous_high and start_idx is not None:
                block = x.loc[start_idx:i]
                episodes.append(Episode(
                    start=block["timestamp"].iloc[0],
                    end=block["timestamp"].iloc[-1],
                    duration_hours=len(block),
                    reward_rentals=int(block["cnt"].sum()),
                ))
                start_idx = None

        return episodes


high_threshold = train["cnt"].quantile(CFG.high_episode_quantile)

extractor = HighDemandEpisodeExtractor(high_threshold)
episodes = extractor.extract(df)

episode_df = pd.DataFrame([e.__dict__ for e in episodes]).sort_values("start").reset_index(drop=True)
episode_df["gap_hours"] = episode_df["start"].diff().dt.total_seconds() / 3600.0

print(f"High-demand threshold: {high_threshold:.1f} rentals/hour")
print(f"Episodes found: {len(episode_df):,}")
display(episode_df.head(10))

High-demand threshold: 413.0 rentals/hour
Episodes found: 817


,start,end,duration_hours,reward_rentals,gap_hours
0,2011-04-11 17:00:00,2011-04-11 17:00:00,1,452,NaN
1,2011-04-18 17:00:00,2011-04-18 17:00:00,1,428,168.0
2,2011-04-19 18:00:00,2011-04-19 18:00:00,1,421,25.0
3,2011-04-20 17:00:00,2011-04-20 18:00:00,2,873,23.0
4,2011-04-21 17:00:00,2011-04-21 18:00:00,2,933,24.0
5,2011-04-23 16:00:00,2011-04-23 16:00:00,1,426,47.0
6,2011-04-24 13:00:00,2011-04-24 14:00:00,2,861,21.0
7,2011-04-25 17:00:00,2011-04-25 18:00:00,2,1020,28.0
8,2011-04-26 08:00:00,2011-04-26 08:00:00,1,449,15.0
9,2011-04-26 17:00:00,2011-04-26 18:00:00,2,1049,9.0


In [36]:
gaps = episode_df["gap_hours"].dropna().to_numpy(dtype=float)
gaps = gaps[gaps > 0]

if gaps.size < 5:
    raise ValueError("Too few positive inter-episode gaps for renewal-distribution fitting.")

gap_mean = gaps.mean()
gap_sd = gaps.std(ddof=1)
gap_cv = gap_sd / gap_mean

# Exponential fit constrained to start at zero.
exp_loc, exp_scale = stats.expon.fit(gaps, floc=0)

# Gamma is a flexible renewal alternative.
gamma_shape, gamma_loc, gamma_scale = stats.gamma.fit(gaps, floc=0)

def loglik_exponential(x):
    return np.sum(stats.expon.logpdf(x, loc=0, scale=exp_scale))

def loglik_gamma(x):
    return np.sum(stats.gamma.logpdf(
        x,
        a=gamma_shape,
        loc=0,
        scale=gamma_scale,
    ))

# k parameters: exponential scale = 1; gamma shape + scale = 2.
aic_exp = 2 * 1 - 2 * loglik_exponential(gaps)
aic_gamma = 2 * 2 - 2 * loglik_gamma(gaps)

renewal_fit = pd.DataFrame({
    "quantity": [
        "mean gap (hours)",
        "gap SD (hours)",
        "gap coefficient of variation",
        "exponential scale",
        "gamma shape",
        "gamma scale",
        "AIC exponential",
        "AIC gamma",
    ],
    "value": [
        gap_mean,
        gap_sd,
        gap_cv,
        exp_scale,
        gamma_shape,
        gamma_scale,
        aic_exp,
        aic_gamma,
    ],
})

display(renewal_fit.round(4))

show(
    PlotFactory.histogram_with_pdfs(
        gaps,
        pdfs={
            "Exponential fit": lambda x: stats.expon.pdf(x, loc=0, scale=exp_scale),
            "Gamma renewal fit": lambda x: stats.gamma.pdf(
                x, a=gamma_shape, loc=0, scale=gamma_scale
            ),
        },
        title="Inter-episode gaps: homogeneous Poisson versus general renewal model",
        x_label="Hours between High-demand episode starts",
        bins=35,
    )
)

,quantity,value
0,mean gap (hours),18.2243
1,gap SD (hours),22.5147
2,gap coefficient of variation,1.2354
3,exponential scale,18.2243
4,gamma shape,2.2657
5,gamma scale,8.0434
6,AIC exponential,6371.2944
7,AIC gamma,6122.0316


## Interpretation

The exponential model is not automatically "the answer."

Instead ask:

1. Is $CV$ near 1?
2. Does the exponential curve resemble the empirical gap distribution?
3. Does a more flexible renewal distribution fit better?
4. Is there obvious 24-hour periodicity?

If inter-event gaps have strong daily structure, the IID renewal assumption is itself imperfect.

This teaches a broader modelling lesson:

$$
\boxed{
\text{A theorem is only as useful as the assumptions connecting it to the data.}
}
$$

---

# 12. Renewal-reward theorem

Suppose each cycle has length $X_i$ and reward $R_i$.

Then under standard conditions,

$$
\frac{Y(t)}{t}
\rightarrow
\frac{E[R]}{E[X]}.
$$

For our event construction:

- cycle length = hours from one High-demand episode start to the next;
- reward = total rentals during the High-demand episode beginning that cycle.

We compare the renewal-reward estimate with the directly observed long-run reward rate.

In [37]:
cycle_df = episode_df.iloc[:-1].copy()
cycle_df["cycle_hours"] = (
    episode_df["start"].shift(-1).iloc[:-1].to_numpy()
    - cycle_df["start"].to_numpy()
) / np.timedelta64(1, "h")

cycle_df = cycle_df[cycle_df["cycle_hours"] > 0].copy()

renewal_reward_rate = (
    cycle_df["reward_rentals"].mean()
    / cycle_df["cycle_hours"].mean()
)

observation_hours = (
    (df["timestamp"].max() - df["timestamp"].min()) / pd.Timedelta(hours=1)
) + 1.0
realized_episode_reward_rate = (
    episode_df["reward_rentals"].sum()
    / observation_hours
)

reward_summary = pd.DataFrame({
    "metric": [
        "E[episode reward]",
        "E[cycle length hours]",
        "E[R] / E[X]",
        "realized episode reward / observed hour",
    ],
    "value": [
        cycle_df["reward_rentals"].mean(),
        cycle_df["cycle_hours"].mean(),
        renewal_reward_rate,
        realized_episode_reward_rate,
    ],
})

display(reward_summary.round(4))

,metric,value
0,E[episode reward],1468.7328
1,E[cycle length hours],18.2243
2,E[R] / E[X],80.5922
3,realized episode reward / observed hour,68.3401


# 13. Brownian motion and the functional central-limit idea

Raw bike demand is not Brownian motion.

Instead we use the Poisson model to remove a large part of deterministic structure and form standardized innovations:

$$
r_t
=
\frac{Y_t-\hat\lambda_t}
{\sqrt{\hat\lambda_t}}.
$$

After centering and standardizing these residuals, define the partial-sum process

$$
W_n(t)
=
\frac{1}{\sqrt n}
\sum_{k=1}^{\lfloor nt\rfloor} Z_k.
$$

Under suitable weak-dependence and finite-variance conditions, a functional central-limit theorem can make such partial sums converge toward Brownian motion:

$$
W_n(\cdot)\Rightarrow B(\cdot).
$$

This is the data-oriented intuition behind Brownian motion as a universal diffusion limit of many accumulated small shocks.

We are **diagnosing the analogy**, not assuming it.

In [38]:
class BrownianResidualAnalyzer:
    def __init__(self, observed: Sequence[float], fitted_rate: Sequence[float]):
        observed = np.asarray(observed, dtype=float)
        fitted_rate = np.clip(np.asarray(fitted_rate, dtype=float), 1e-9, None)

        if observed.shape != fitted_rate.shape:
            raise ValueError("observed and fitted_rate must have the same shape.")
        if observed.size < 2:
            raise ValueError("At least two residual observations are required.")

        pearson = (observed - fitted_rate) / np.sqrt(fitted_rate)
        sd = pearson.std(ddof=1)
        if not np.isfinite(sd) or np.isclose(sd, 0.0):
            raise ValueError("Residual standard deviation is zero or non-finite.")
        self.z = (pearson - pearson.mean()) / sd

    def functional_partial_sum(self) -> tuple[np.ndarray, np.ndarray]:
        n = len(self.z)
        t = np.arange(1, n + 1) / n
        w = np.cumsum(self.z) / np.sqrt(n)
        return t, w

    def block_variance_scaling(self, block_sizes: Sequence[int]) -> pd.DataFrame:
        rows = []

        for size in block_sizes:
            n_blocks = len(self.z) // size
            if n_blocks < 5:
                continue

            trimmed = self.z[: n_blocks * size]
            block_sums = trimmed.reshape(n_blocks, size).sum(axis=1)

            rows.append({
                "block_size": size,
                "variance_of_block_sum": block_sums.var(ddof=1),
                "brownian_reference": float(size),
                "variance_ratio_to_size": block_sums.var(ddof=1) / size,
            })

        return pd.DataFrame(rows)


brownian_data = BrownianResidualAnalyzer(
    test["cnt"],
    test["lambda_hat"],
)

t_emp, w_emp = brownian_data.functional_partial_sum()

# Simulated standard Brownian path on the same [0,1] grid.
n = len(t_emp)
brownian_increments = RNG.normal(
    loc=0,
    scale=np.sqrt(1 / n),
    size=n,
)
w_sim = np.cumsum(brownian_increments)

show(
    PlotFactory.multi_line(
        x=t_emp,
        series={
            "Accumulated standardized demand innovations": w_emp,
            "One standard Brownian path": w_sim,
        },
        title="Functional-CLT view: empirical partial sums versus Brownian motion",
        x_label="Rescaled time t in [0,1]",
        y_label="Scaled cumulative innovation",
    )
)

In [39]:
scaling = brownian_data.block_variance_scaling(
    block_sizes=[1, 2, 4, 8, 12, 24, 48, 72, 120]
)

display(scaling.round(4))

show(
    PlotFactory.multi_line(
        x=scaling["block_size"].to_numpy(),
        series={
            "Empirical variance of block sum": scaling["variance_of_block_sum"].to_numpy(),
            "Brownian / independent-increment reference": scaling["brownian_reference"].to_numpy(),
        },
        title="Variance scaling of accumulated demand innovations",
        x_label="Aggregation window length",
        y_label="Variance of summed standardized innovations",
    )
)

,block_size,variance_of_block_sum,brownian_reference,variance_ratio_to_size
0,1,1.0000,1.0,1.0000
1,2,3.5167,2.0,1.7584
2,4,10.7170,4.0,2.6793
3,8,22.7792,8.0,2.8474
4,12,26.1317,12.0,2.1776
5,24,73.2114,24.0,3.0505
6,48,247.2637,48.0,5.1513
7,72,497.6646,72.0,6.9120
8,120,1165.7163,120.0,9.7143


## Why the variance-scaling plot matters

For independent standardized increments,

$$
\operatorname{Var}
\left(
\sum_{k=1}^{m}Z_k
\right)
=
m.
$$

Brownian increments have exactly analogous linear variance growth:

$$
\operatorname{Var}(B_{t+h}-B_t)=h.
$$

Therefore the reference line is proportional to window length.

Persistent serial correlation causes the empirical variance to deviate from that line.

---

# 14. Brownian first passage and the reflection principle

For standard Brownian motion,

$$
M_T
=
\max_{0\le t\le T}B_t.
$$

The reflection principle gives

$$
P(M_T\ge a)
=
2P(B_T\ge a).
$$

Since

$$
B_T\sim N(0,T),
$$

we obtain

$$
\boxed{
P(\tau_a\le T)
=
2\left[
1-\Phi\left(\frac{a}{\sqrt T}\right)
\right]
}
$$

where

$$
\tau_a
=
\inf\{t:B_t=a\}.
$$

We can verify this theorem numerically.

In [21]:
class BrownianFirstPassageExperiment:
    def __init__(self, rng: np.random.Generator):
        self.rng = rng

    def run(
        self,
        level: float,
        T: float,
        n_paths: int,
        n_steps: int,
    ) -> dict:
        if level <= 0:
            raise ValueError("level must be positive for this first-passage experiment.")
        if T <= 0 or n_paths <= 0 or n_steps <= 0:
            raise ValueError("T, n_paths and n_steps must all be positive.")

        dt = T / n_steps
        current = np.zeros(n_paths, dtype=float)
        hit = np.zeros(n_paths, dtype=bool)

        # Streaming simulation: O(n_paths) memory instead of O(n_paths*n_steps).
        for _ in range(n_steps):
            current += self.rng.normal(0.0, np.sqrt(dt), size=n_paths)
            hit |= current >= level

        empirical = float(hit.mean())
        theoretical = float(2 * (1 - stats.norm.cdf(level / np.sqrt(T))))

        return {
            "level": level,
            "T": T,
            "paths": n_paths,
            "steps_per_path": n_steps,
            "empirical_hit_probability": empirical,
            "reflection_principle_probability": theoretical,
            "absolute_error": abs(empirical - theoretical),
        }


fp_experiment = BrownianFirstPassageExperiment(
    np.random.default_rng(CFG.random_seed)
)

fp_result = fp_experiment.run(
    level=1.5,
    T=1.0,
    n_paths=CFG.brownian_mc_paths,
    n_steps=CFG.brownian_mc_steps,
)

display(pd.DataFrame([fp_result]).round(5))

,level,T,paths,steps_per_path,empirical_hit_probability,reflection_principle_probability,absolute_error
0,1.5,1.0,5000,500,0.132,0.13361,0.00161


### Monte Carlo complexity

With $B$ simulated paths and $m$ time steps per path:

$$
\Theta(Bm)
$$

random increments are generated.

The implementation above streams through time, so memory is only $O(B)$: it stores the current value of each path and a Boolean "already hit" vector rather than the complete $B\times m$ path matrix.

---

# 15. Factory / Registry pattern for stochastic analysers

As the notebook grows, we may want to construct analysers by symbolic name.

The registry below keeps orchestration code independent of concrete classes.

In [40]:
class AnalysisRegistry:
    _builders: Dict[str, Type] = {}

    @classmethod
    def register(cls, name: str):
        def decorator(analyser_cls: Type):
            cls._builders[name] = analyser_cls
            return analyser_cls
        return decorator

    @classmethod
    def create(cls, name: str, *args, **kwargs):
        if name not in cls._builders:
            raise KeyError(
                f"Unknown analyser '{name}'. "
                f"Available: {sorted(cls._builders)}"
            )
        return cls._builders[name](*args, **kwargs)

    @classmethod
    def available(cls) -> list[str]:
        return sorted(cls._builders)


# Register existing reusable implementations through the public decorator API.
for _name, _cls in {
    "markov": FiniteMarkovChain,
    "ctmc": CTMCFromSnapshots,
    "poisson_demand": PoissonDemandModel,
    "brownian_residual": BrownianResidualAnalyzer,
}.items():
    AnalysisRegistry.register(_name)(_cls)

print("Registered analysers:", AnalysisRegistry.available())

# Example construction:
demo_chain = AnalysisRegistry.create(
    "markov",
    state_labels=regime_strategy.labels,
)
type(demo_chain)

Registered analysers: ['brownian_residual', 'ctmc', 'markov', 'poisson_demand']


__main__.FiniteMarkovChain

# 16. Assumption scorecard

A useful stochastic modeller does not merely fit models. They ask whether the mechanism is credible.

The table below summarizes the principal assumptions and what to inspect.

In [41]:
scorecard = pd.DataFrame([
    {
        "model": "Finite DTMC",
        "assumption": "Next regime depends sufficiently on current regime; approximately time homogeneous.",
        "diagnostic_or_warning": "Compare transition matrices by hour/weekday/season; strong differences imply non-homogeneity.",
    },
    {
        "model": "CTMC",
        "assumption": "Exponential-like holding structure and continuously observed jumps.",
        "diagnostic_or_warning": "We only have hourly snapshots; Q is an approximation and hidden within-hour jumps are unobserved.",
    },
    {
        "model": "Poisson counts",
        "assumption": "Conditional mean approximately equals conditional variance; suitable conditional independence.",
        "diagnostic_or_warning": f"Pearson dispersion = {poisson_model.pearson_dispersion():.2f}. Large values indicate overdispersion.",
    },
    {
        "model": "Renewal",
        "assumption": "Inter-renewal gaps are approximately IID.",
        "diagnostic_or_warning": f"Gap CV = {gap_cv:.2f}; daily periodicity can violate IID/exponential assumptions.",
    },
    {
        "model": "Brownian scaling limit",
        "assumption": "Centered innovations have finite variance and sufficiently weak dependence.",
        "diagnostic_or_warning": "Use block-variance scaling and residual autocorrelation before treating innovations as diffusion-like.",
    },
])

display(scorecard)
show(PlotFactory.safe_table(scorecard, "Stochastic-model assumption scorecard", height=360))

,model,assumption,diagnostic_or_warning
0,Finite DTMC,Next regime depends sufficiently on current re...,Compare transition matrices by hour/weekday/se...
1,CTMC,Exponential-like holding structure and continu...,We only have hourly snapshots; Q is an approxi...
2,Poisson counts,Conditional mean approximately equals conditio...,Pearson dispersion = 38.95. Large values indic...
3,Renewal,Inter-renewal gaps are approximately IID.,Gap CV = 1.24; daily periodicity can violate I...
4,Brownian scaling limit,Centered innovations have finite variance and ...,Use block-variance scaling and residual autoco...


# 17. A time-inhomogeneous extension

The simple Markov model assumes one transition matrix $P$ for all hours.

Bike demand is strongly time dependent. A more realistic extension estimates

$$
P^{(h)}
$$

separately for each hour of the day.

Then

$$
P(X_{t+1}=j\mid X_t=i,\text{hour}=h)
=
P^{(h)}_{ij}.
$$

This is a **time-inhomogeneous Markov chain**.

The next cell compares the probability of moving to High demand from Medium demand across the day.

In [42]:
def hourly_transition_matrices(
    frame: pd.DataFrame,
    labels: Sequence[str],
) -> dict[int, np.ndarray]:
    state_to_idx = {s: i for i, s in enumerate(labels)}
    m = len(labels)
    counts = {h: np.zeros((m, m), dtype=int) for h in range(24)}

    x = frame.sort_values("timestamp").reset_index(drop=True)

    for k in range(len(x) - 1):
        if x.loc[k + 1, "timestamp"] - x.loc[k, "timestamp"] != pd.Timedelta(hours=1):
            continue

        h = int(x.loc[k, "hr"])
        i = state_to_idx[str(x.loc[k, "regime"])]
        j = state_to_idx[str(x.loc[k + 1, "regime"])]
        counts[h][i, j] += 1

    matrices = {}
    for h, c in counts.items():
        row_sum = c.sum(axis=1, keepdims=True)
        matrices[h] = np.divide(
            c,
            row_sum,
            out=np.full_like(c, np.nan, dtype=float),
            where=row_sum > 0,
        )

    return matrices


hourly_P = hourly_transition_matrices(train, markov.state_labels)
medium_idx = markov.state_to_idx["Medium"]
high_idx = markov.state_to_idx["High"]

medium_to_high = np.array([
    hourly_P[h][medium_idx, high_idx]
    for h in range(24)
])

p = figure(
    width=900,
    height=350,
    title="Time-inhomogeneity: P(Medium → High) by hour",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p.line(np.arange(24), medium_to_high, line_width=2)
p.scatter(np.arange(24), medium_to_high, size=7)
p.xaxis.axis_label = "Current hour"
p.yaxis.axis_label = "Estimated transition probability"
show(p)

## Why this extension matters

The homogeneous Markov assumption says

$$
P(X_{t+1}\mid X_t)
$$

does not change with calendar time.

A time-inhomogeneous chain allows

$$
P(X_{t+1}\mid X_t,t).
$$

This is often much more realistic for:

- traffic;
- electricity demand;
- server workloads;
- customer activity;
- epidemiological states.

However, the price of flexibility is more parameters and fewer observations per transition matrix.

That is the classic statistical trade-off:

$$
\boxed{
\text{bias from oversimplification}
\quad\leftrightarrow\quad
\text{variance from overparameterization}.
}
$$

---

# 18. Empirical validation of the stationary distribution

The stationary distribution is a theoretical long-run object.

We can compare it with the empirical state proportions in a long held-out period.

For a genuinely stationary ergodic chain,

$$
\frac{1}{n}\sum_{t=1}^{n}\mathbf 1(X_t=j)
\rightarrow
\pi_j.
$$

Because bike demand is seasonal and non-homogeneous, disagreement is informative rather than merely "wrong."

In [25]:
empirical_test_pi = (
    test["regime"]
    .astype(str)
    .value_counts(normalize=True)
    .reindex(markov.state_labels)
    .to_numpy()
)

stationary_compare = pd.DataFrame({
    "state": markov.state_labels,
    "training_chain_stationary": pi_eigen,
    "held_out_empirical_fraction": empirical_test_pi,
})
stationary_compare["difference"] = (
    stationary_compare["held_out_empirical_fraction"]
    - stationary_compare["training_chain_stationary"]
)

display(stationary_compare.round(4))

,state,training_chain_stationary,held_out_empirical_fraction,difference
0,Low,0.3270,0.2699,-0.0571
1,Medium,0.3426,0.2385,-0.1041
2,High,0.3304,0.4917,0.1612


# 19. What each ST5221 theory contributed

| ST5221 idea | Case-study interpretation | Main mathematical object |
|---|---|---|
| Stochastic process | Hourly demand evolving randomly | $\{X_t\}$ |
| DTMC | Low/Medium/High demand transitions | $P$ |
| Chapman–Kolmogorov | Multi-hour transition probabilities | $P^n$ |
| Communicating classes | Which regimes can reach one another? | directed graph |
| Stationarity | Long-run regime occupancy | $\pi P=\pi$ |
| Limit theorem | Forgetting the initial regime | $P^n\to\mathbf 1\pi$ |
| First passage | Time until High demand | $(I-Q)m=\mathbf 1$ |
| CTMC | Approximate transition rates | generator $Q$ |
| Kolmogorov equations | Continuous-time propagation | $P(t)=e^{Qt}$ |
| Poisson process | Count/intensity approximation | $\lambda_t$ |
| Renewal process | High-demand episode starts | $S_n=\sum X_i$ |
| Renewal reward | Long-run episode-associated rentals | $E[R]/E[X]$ |
| Brownian motion | Scaling limit for accumulated innovations | $B_t$ |
| Reflection principle | First-passage probability | $P(M_T\ge a)$ |

---

# 20. Suggested experiments

These are useful extensions if you want to turn the notebook into a larger project.

### Experiment A — Alternative state spaces

Replace quantile states with:

- 5 demand regimes;
- K-means states over `cnt`, temperature and humidity;
- manually defined operational thresholds.

Study how the spectral gap and hitting times change.

### Experiment B — Context-specific Markov chains

Estimate separate $P$ matrices for:

- working versus non-working days;
- clear versus rainy weather;
- morning versus evening;
- summer versus winter.

Compare stationary distributions and mixing.

### Experiment C — Semi-Markov modelling

The Markov chain implicitly makes duration behaviour geometric in discrete time.

Empirically estimate regime-duration distributions.

If duration depends strongly on elapsed time, a **semi-Markov model** is more appropriate.

### Experiment D — Non-homogeneous Poisson intensity

Replace categorical hour effects with a cyclic spline or Fourier basis:

$$
\log\lambda_t
=
\beta_0+
a_1\sin(2\pi h/24)+
b_1\cos(2\pi h/24)+\cdots.
$$

### Experiment E — Renewal theory by context

Extract High-demand episode starts separately for working days and weekends.

This can reduce mixing of different stochastic mechanisms.

### Experiment F — Brownian diagnostic

Remove additional autocorrelation from the Poisson residuals and repeat the functional-CLT diagnostic.

You should expect Brownian-like behaviour to improve only when the residual innovations are sufficiently weakly dependent.

# 21. Final conceptual synthesis

This case study illustrates a central lesson of stochastic processes:

$$
\boxed{
\text{Random microscopic evolution can create stable macroscopic laws.}
}
$$

A particular bike-demand path remains unpredictable, yet stochastic-process theory lets us reason about:

- transition probabilities;
- long-run occupancy;
- first-passage times;
- continuous-time transition rates;
- event intensities;
- recurrence intervals;
- long-run reward rates;
- diffusion limits.

The modelling workflow is:

$$
\boxed{
\text{Define the stochastic state}
\rightarrow
\text{state assumptions}
\rightarrow
\text{estimate dynamics}
\rightarrow
\text{derive theoretical quantities}
\rightarrow
\text{validate assumptions}
}
$$

The final step is essential.

A Markov chain, Poisson process, renewal process or Brownian model is not valuable because it has elegant mathematics alone. It is valuable when the assumptions create a useful approximation to the mechanism that generated the data.

---

## References

- NUS Department of Statistics & Data Science — ST5221 Stochastic Processes and Applications.
- UCI Machine Learning Repository — Bike Sharing Dataset.
- Feller, *An Introduction to Probability Theory and Its Applications*.
- Ross, *Stochastic Processes*.
- Norris, *Markov Chains*.